# Probability of Fire (PoF) — Data Generator

**Master:** Physics of Data \
**Course:** Laboratory of Computational Physics (LCP), Module B \
**Authors:** Gabriela Landinez Rangel, Andres Rojas Lozano, Fatemeh Dashti, Arash Taraz Jamshidi

*This notebook was created by us to present the final project for LCP MOD B, However it is based on public available code from the Probability of Fire project by ECMWF: https://ecmwf.github.io/AI-Probability-of-Fire/pof-forecast/. With this work, we aim to provide a clear, but friendly, guide with straightforward instructions and detailed explanations for students who, like us, want to reproduce this framework.*

## Python libraries

In [2]:
import xarray as xr       #to open and manipulate NetCDF files.(standard file format used in gridded climate data)
import numpy as np
import pandas as pd       #XGBOOST works with a dataframe: where each row corresponds to one grid cell and one day.
import joblib             #to load the trained model
from pathlib import Path
import os
import cftime             #to time handling in NetCDF files
from xarray.coding.times import CFDatetimeCoder
import cdsapi             #to download weather data: from the Copernicus Climate Data Store
import zipfile
import dask               #to help with larger datasets, parallel processing
from dask import delayed, compute
from dask.diagnostics import ProgressBar

## the configuration of the forecasting experiment

##### First, we define the data folder:data
* This means the notebook expects all input files, including the trained model and downloaded datasets, to be inside the data folder.
* Then we create the outputs folder if it does not already exist.


##### Then we define the forecast period(December 2019) and the output filename. Then we load the trained model.

In [3]:
# Folder structure
base_path = Path("./data/")                 #define the data folder(trained model + prediction dataset)
os.makedirs("./outputs/", exist_ok=True)    #create the outputs folder(final file)


# Dates of Forecast(forecast period)
year = 2019
years = [year]
months = [12]
cds_months=[f'{m:02d}' for m in months]      #converts the month number into a two-digit string(needed by the CDS request)
cds_days= [f"{d:02d}" for d in range(1, 32)] #creates the list of days from "01" to "31"


# output file name, Where we will store our predictions
output_file = f"./outputs/POF_prediction_{year}_{months[0]:02d}.nc"

# Load trained model
model = joblib.load("./data/POF_model.joblib")


## Access to forecast information

In [4]:
# download ERA5-Land weather data, such as temperature, dew point, wind components, and precipitation.
cds_kwargs = {
    'url': 'https://cds.climate.copernicus.eu/api',
    'key': '7d185cca-7153-44d6-b716-0a154a2e96b1',
}


# download fuel datasets
xds_kwargs = {
    'url': 'https://xds-preprod.ecmwf.int/api',
    'key': 'e26e7334-f401-4c4b-b8cf-8dc193987dfd',
}

## Download weather data + create derived variables(WS and RH)
This code defines cds_kwargs, xds_kwargs, downloads ERA5-Land weather variables, creates WS, and derives relative humidity RH from temperature and dew-point temperature.


In [5]:
#loop over the selected year and month
for year in years:
    for month in cds_months: 
        #This loop defines the weather variables that we want to download.(shortname:filename)(var:real variable name requested from ERA5-Land)
        for shortname,var in [["T2M","2m_temperature"],    #near surface temperature
                      ["D2M","2m_dewpoint_temperature"],   #dew point temperature
                      ["10U","10m_u_component_of_wind"],   #east_west wind component
                      ["10V","10m_v_component_of_wind"],   #north_south wind component
                      ["P","total_precipitation"]]:        #precipitation

            #Defining the CDS request: sent to the Copernicus Climate Data Store
            dataset = "reanalysis-era5-land"
            request = {
                "variable": [
                    var,   #means: each request downloads one variable at a time
                ],
                "year": year,
                "month": month,
                "day": cds_days,
                "time": [
                    "00:00"     #at midnight
                ],
                "data_format": "netcdf",      #ile is saved in NetCDF format
                "download_format": "unarchived"
            }

            #Download (only if the file does not already exist)
            #This creates the CDS client and downloads the data.
            client = cdsapi.Client(**cds_kwargs)
            if not os.path.exists(f"data/{shortname}_{year}_{month}.nc"):
                client.retrieve(dataset, request, target=f"data/{shortname}_{year}_{month}.nc")


        #Creating the wind-speed feature (ERA5-Land does not directly provide the WS)
        if not os.path.exists(f"data/WS_{year}_{month}.nc"):
            with xr.open_dataset(f"data/10U_{year}_{month}.nc") as dsu, \
                xr.open_dataset(f"data/10V_{year}_{month}.nc") as dsv:      
                dsw=dsu.u10**2+ dsv.v10**2                    #combines 2 wind components into one wind variable                
                dsw=dsw.rename("ws")
                dsw.to_netcdf(f"data/WS_{year}_{month}.nc")
                del dsw
            #u10 = dsu = east-west wind component 
            #v10 = dsv = north-south wind component
            

        
        #Creating relative humidity
        if not os.path.exists(f"data/RH_{year}_{month}.nc"):
            with xr.open_dataset(f"data/T2M_{year}_{month}.nc") as dst,\
                xr.open_dataset(f"data/D2M_{year}_{month}.nc") as dsd:      
                es = 10**(7.5*(dst.t2m-273.15)/(237.7+(dst.t2m-273.15))) * 6.11     #saturation vapour pressure (using air temperature)
                e =  10**(7.5*(dsd.d2m-273.15)/(237.7+(dsd.d2m-273.15))) * 6.11     #actual vapour pressure (using dew-point temperature)
                dsrh = (e/es)*100                 #relative humidity(tells us how close the air is to saturation)
                dsrh = dsrh.rename("rh")
                dsrh.to_netcdf(f"data/RH_{year}_{month}.nc")
                del dsrh
            #dst= near surface temeprature
            #dsd= dew point temperature

## Downloading and regridding fuel-related data
* This cell uses Dask chunks, @delayed, and parallel execution.
* it opens a latitude/longitude reference dataset, downloads DFMC_MAP, LFMC_MAP, and FUEL_MAP from the derived-fire-fuel-biomass dataset, extracts the zip files, interpolates them to the reference grid with interp_like, and saves regridded files with _R.nc.
* The main idea is:
fuel datasets may not have exactly the same latitude-longitude grid as the weather data, so we regrid them before using them as model inputs.

In [6]:
# Open lat-lon reference dataset with chunks for Dask
lat_lon_data = xr.open_dataset(
    f"data/T2M_{years[0]}_{cds_months[0]}.nc",   #So T2M_2019_12.nc becomes the reference grid.
    chunks={'latitude': 100, 'longitude': 100}  # adjust chunk size to your RAM(It tells xarray/Dask to process the data in smaller spatial)
).sortby("latitude").sortby("longitude")

lat_lon_data = lat_lon_data.rename(valid_time="time", latitude='lat', longitude='lon')
lat_lon_data = lat_lon_data.drop_vars(['expver', 'number'])       #Drop unnecessary variables(metadata)




#Define a delayed function: The function will process one variable for one month and one year.
# Function to download and regrid a single variable/month/year
#after this, we can run the 3 downloads/regridding tasks in parallel.
@delayed
def process_variable(year, month, shortname, var):
    dataset = "derived-fire-fuel-biomass"
    request = {
        "variable": [var],
        "version": ["1"],
        "year": [f"{year}"],
        "month": [month]
    }

    #Define file paths(for example for FUEL_MAP fisrt)
    client = cdsapi.Client(**xds_kwargs)
    zip_path = f"data/{shortname}_{year}_{month}.zip"   #becomes: compressed file downloaded from the server
    nc_path = f"data/{shortname}_{year}_{month}.nc"     #extracted NetCDF file
    regrid_path = f"data/{shortname}_{year}_{month}_R.nc"  #regridded NetCDF file

    if os.path.exists(regrid_path):
        return regrid_path  # Already processed

    
    # Download and extract the file: 
    #downloads as a zip file, then extracts the NetCDF file inside the data/ folder.
    client.retrieve(dataset, request, target=zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall("data/")

    print(f"Regridding {shortname} for {year}-{month}")

    #Regrid the fuel dataset ---> onto the same grid as the weather dataset
    # Open with Dask chunks and interpolate
    with xr.open_dataset(nc_path, chunks={'latitude': 100, 'longitude': 100}) as ds_temp:       #ds_temp is the original fuel dataset
        ds_regrid = ds_temp.interp_like(lat_lon_data, method="linear")       #interpolates the fuel dataset onto the same grid as the weather dataset.
        ds_regrid.to_netcdf(regrid_path)
        del ds_regrid   

    return regrid_path




# Build delayed tasks for all combinations of fuel datasets
tasks = []
for year in years:
    for month in cds_months:
        for shortname, var in [
            ["DFMC_MAP", "dead_fuel_moisture_content_group"],
            ["LFMC_MAP", "live_fuel_moisture_content_group"],
            ["FUEL_MAP", "fuel_group"]
        ]:
            tasks.append(process_variable(year, month, shortname, var))

# Execute tasks in parallel, using Dask (all cores)
results = compute(*tasks, scheduler='threads')  # use 'processes' if memory allows

C:\Users\ASUS\AppData\Local\Temp\ipykernel_7824\2879459735.py:2: UserWarning: The specified chunks separate the stored chunks along dimension "latitude" starting at index 100. This could degrade performance. Instead, consider rechunking after loading.
  lat_lon_data = xr.open_dataset(
C:\Users\ASUS\AppData\Local\Temp\ipykernel_7824\2879459735.py:2: UserWarning: The specified chunks separate the stored chunks along dimension "longitude" starting at index 100. This could degrade performance. Instead, consider rechunking after loading.
  lat_lon_data = xr.open_dataset(


## Loading static climate data (population and Road density)
* static ignition-proxy data do not change day by day. 

* Weather and fuel moisture change daily, but population density and road density are treated as fixed background variables for this example.

In [7]:
# Open the Climate files
time_coder = CFDatetimeCoder(use_cftime=True)  #creates a time decoder for climate-style dates(optional)
PO = xr.open_dataset(base_path / "CLIMATE/POP_2020.nc")
RD = xr.open_dataset(base_path / "CLIMATE/road_density_2015_agg_r.nc")

# Extract the arrays
PO_arr = PO.population_density.values  
RD_arr = RD.road_length.values         

## Building predictions
* this cell opens all dynamic datasets
* loops through each day, builds the prediction dataframe ---> runs model.predict_proba ---> rebuilds the probability grid
* and saves the result as POF_prediction_2019_12.nc.

In [8]:
for year in years:
    for month in months:
        # Define the paths of the files...
        #This dictionary: stores the names of all files needed for prediction.
        ds_paths = {
            "AF": f"ACTIVE_FIRE_MAP_{year}_{month:02d}_R.nc",
            "FU": f"FUEL_MAP_{year}_{month:02d}_R.nc",
            "DF": f"DFMC_MAP_{year}_{month:02d}_R.nc",
            "LF": f"LFMC_MAP_{year}_{month:02d}_R.nc",
            "PR": f"P_{year}_{month:02d}.nc",
            "T2": f"T2M_{year}_{month:02d}.nc",
            "RH": f"RH_{year}_{month:02d}.nc",
            "WS": f"WS_{year}_{month:02d}.nc",
        }
        # Open all dynamic datasets
        with xr.open_dataset(base_path / ds_paths["FU"]) as FU, \
            xr.open_dataset(base_path / ds_paths["DF"]) as DF, \
            xr.open_dataset(base_path / ds_paths["LF"]) as LF, \
            xr.open_dataset(base_path / ds_paths["PR"]) as PR, \
            xr.open_dataset(base_path / ds_paths["T2"]) as T2, \
            xr.open_dataset(base_path / ds_paths["RH"]) as RH, \
            xr.open_dataset(base_path / ds_paths["WS"]) as WS, \
            xr.open_dataset(base_path / ds_paths["AF"]) as AF:

            n_days = len(AF.ACTIVE_FIRE)
            all_grids = []                 #one probability map per day will be appended.


            #Loop over days:
            #This loop processes one day at a time.
            #For each day, it extracts the weather, fuel, and moisture maps--> creates the prediction dataframe-->predicts probability--> and reconstructs the daily map.    
            for i in range(n_days):
                # Extract arrays for timestep i (for one day)
                FU_LL = FU.Live_Leaf[i].values
                FU_LW = FU.Live_Wood[i].values
                FU_DF = FU.Dead_Foliage[i].values
                FU_DW = FU.Dead_Wood[i].values
                DF_ = DF.DFMC_Foliage[i].values
                DW_ = DF.DFMC_Wood[i].values
                LF_ = LF.LFMC[i].values
                PR_ = PR.tp[i].values
                T2_ = T2.t2m[i].values
                RH_ = RH.rh[i].values
                WS_ = WS.ws[i].values

                # Mask where total fuel > 0
                ft = FU_LL + FU_LW + FU_DF + FU_DW
                mask = ft > 0

                # Flatten and build dataframe for prediction(key step)
                #The original data are maps. But XGBoost expects a table.
                feature_arrays = {
                    "PR": PR_[mask],
                    "T2": T2_[mask],
                    "RH": RH_[mask],
                    "WS": WS_[mask],
                    "FU_LL": FU_LL[mask],
                    "FU_LW": FU_LW[mask],
                    "FU_DF": FU_DF[mask],
                    "FU_DW": FU_DW[mask],
                    "DF": DF_[mask],
                    "DW": DW_[mask],
                    "LF": LF_[mask],
                    "PO": PO_arr[mask],
                    "RD": RD_arr[mask],
                }
                X_pred = pd.DataFrame(feature_arrays)
                #This is important because the forecasting dataframe must match the training feature structure

                
                # Predict probability
                y_proba = model.predict_proba(X_pred)[:, 1]
                

                # Rebuild the full grid:
                #The model predicted only for valid fuel cells.
                #Now we need to rebuild the full latitude-longitude map.
                fire_prob_grid = np.full(ft.shape, np.nan, dtype=float)     #creates a full grid filled with NaN.
                fire_prob_grid[mask] = y_proba     #puts the predicted probabilities back into the valid fuel cells.
                all_grids.append(fire_prob_grid)   #Then the daily map is added here.

            # Stack all daily maps (timesteps) into 3D array (time, lat, lon)
            fire_prob_array = np.stack(all_grids, axis=0)
            
            # -----------------------------
            # SAVE TO NETCDF
            # -----------------------------
                
            #Create one time coordinate per day(1st dec, 2nd dec, ... , 31st dec)    
            time = [cftime.DatetimeJulian(year, month, day+1) for day in range(n_days)]

            #Create the final dataset to save
            #contains only one variable(contains one variable)
            ds_out = xr.Dataset(
                {"fire_probability": (["time", "latitude", "longitude"], fire_prob_array)},
                coords={
                    "time": time,
                    "latitude": PO.latitude,
                    "longitude": PO.longitude,
                }
            )

            #If the output file already exists, it removes it first. Then it saves the new output.    
            os.remove(output_file) if os.path.exists(output_file) else None
            ds_out.to_netcdf(output_file)
            print(f"✅ Prediction saved → {output_file}")



✅ Prediction saved → ./outputs/POF_prediction_2019_12.nc
